In [1]:
import pathlib as pl

# adapt relative path until it loads
cfg_nb = pl.Path("../../load-config.ipynb").resolve(strict=True)
%run $cfg_nb

_NB_SESSION = __session__
NB_NAME = pl.Path(_NB_SESSION).name
NB_PATH = pl.Path(_NB_SESSION).parent
NB_REL_PATH = NB_PATH.relative_to(CONFIG["project_repo"]).joinpath(NB_NAME)

PLOT_ROOT = CONFIG["plot_root"].joinpath(NB_PATH.stem)
TABLE_ROOT = CONFIG["table_root"].joinpath(NB_PATH.stem)
NB_CACHE_FOLDER = CONFIG["nb_cache_folder"]
DATA_ROOT = CONFIG["project_data"]

import matplotlib as mpl

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# NB-specific code

import pandas as pd

refs = {
    "t2tv2": "T2Tv2",
    "hg38": "GRCh38"
}

LABEL_FILE_SOURCE = DATA_ROOT.joinpath("wf-data/region-labels/final_annot/2026-02_final-rev")

def assign_seq_type(seqname):
    if seqname.endswith("_chrY"):
        st = "main"
    else:
        st = "rand"
    return st

relevant_labels = "[P13]{2}"

sample_stats = []
for ref in refs.keys():
    label_files = LABEL_FILE_SOURCE.joinpath(ref).glob("*.bed")
    for label_file in label_files:
        sample = label_file.name.split(".")[0]
        df = pd.read_csv(label_file, sep="\t", header=0)
        df.rename({"#seq": "seq"}, axis=1, inplace=True)
        df["seqtype"] = df["seq"].apply(assign_seq_type)
        df["length"] = df["end"] - df["start"]
        stats = {"ref": ref, "sample": sample}
        rand_seq_n = df["seq"].nunique() - 1
        stats["num_rand_seq"] = rand_seq_n
        total_length = int(df.groupby("seq")["end"].max().sum())
        stats["total_length_bp"] = total_length
        select_rand = df.loc[df["seqtype"] == "rand", ]
        if select_rand.empty:
            stats["rand_length_bp"] = 0
        else:
            stats["rand_length_bp"] = int(select_rand.groupby("seq")["end"].max().sum())
        stats["main_contains_motif"] = 0
        stats["rand_contains_motif"] = 0
        stats["randseq_motif_length_bp"] = 0
        for seq, seq_regions in df.groupby("seq"):
            if (seq_regions["name"].str.contains(relevant_labels, regex=True)).any():
                is_main = seq_regions["seqtype"].iloc[0] == "main"
                if is_main:
                    stats["main_contains_motif"] += 1  # this should always be 0 or 1
                else:
                    stats["rand_contains_motif"] += 1
                    this_length = seq_regions["end"].max()
                    stats["randseq_motif_length_bp"] += this_length
        sample_stats.append(stats)

sample_stats = pd.DataFrame.from_records(sample_stats)

for ref, stats in sample_stats.groupby("ref"):
    print("Reference: ", ref)
    samples_w_rand = (stats["num_rand_seq"] > 0).sum()
    total_rand = stats["num_rand_seq"].sum()
    motif_rand = stats["rand_contains_motif"].sum()
    fraction_w_motif = (motif_rand / total_rand * 100).round(1)
    print("Samples (n) w/ rand seqs.: ", samples_w_rand)
    print("Rand. seqs. w/ motif (%): ", fraction_w_motif)
    print(motif_rand, " out of ", total_rand)

            
        
        

        
            
            

Reference:  hg38
Samples (n) w/ rand seqs.:  83
Rand. seqs. w/ motif (%):  20.3
162  out of  799
Reference:  t2tv2
Samples (n) w/ rand seqs.:  83
Rand. seqs. w/ motif (%):  18.8
150  out of  799
